<a href="https://colab.research.google.com/github/Tayyaba1206/FlyRank-Internship/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
import os

print(os.getcwd())
print(os.listdir())

/content
['.config', 'sample_data']


In [7]:
import os, sys, subprocess

REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)

os.chdir(REPO_DIR)

print("Working directory:", os.getcwd())
print(os.path.exists("data/raw/content_refresh_anonymized.csv"))

Working directory: /content/flyrank-ml-internship-starter
True


# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Tayyaba1206/FlyRank-Internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

## My Rule

I will prioritize pages that are old, have high impressions, and have a low CTR. These pages may benefit from content updates because they are still visible but may not be attracting enough clicks.

### Reason Codes

- OLD_CONTENT – The page has not been updated for a long time.
- LOW_CTR – The page receives impressions but has a low click-through rate.
- HIGH_IMPRESSIONS – The page is visible in search results and has optimization potential.

### Action Label

Review and Refresh Content

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [8]:
import pandas as pd
import os

# Load dataset
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

# Create baseline score
df["baseline_score"] = (
    (df["days_since_last_update"] >= 180).astype(int) * 2 +
    (df["ctr"] < 0.10).astype(int) * 2 +
    (df["impressions_90d"] > 500).astype(int)
)

# Reason Codes
def reason(row):
    reasons = []
    if row["days_since_last_update"] >= 180:
        reasons.append("OLD_CONTENT")
    if row["ctr"] < 0.10:
        reasons.append("LOW_CTR")
    if row["impressions_90d"] > 500:
        reasons.append("HIGH_IMPRESSIONS")
    return ", ".join(reasons)

df["reason_code"] = df.apply(reason, axis=1)

# Action Label
df["action"] = "Review and Refresh Content"

# Rank pages
df = df.sort_values("baseline_score", ascending=False)

# Save CSV
os.makedirs("work/outputs", exist_ok=True)
df.to_csv("work/outputs/baseline_action_score.csv", index=False)

print("CSV saved successfully!")

# Show top 20
df.head(20)[["content_id","baseline_score","reason_code","action"]]

CSV saved successfully!


,content_id,baseline_score,reason_code,action
698,content_b16bd7307b39,5,"OLD_CONTENT, LOW_CTR, HIGH_IMPRESSIONS",Review and Refresh Content
11489,content_5feee3994adb,5,"OLD_CONTENT, LOW_CTR, HIGH_IMPRESSIONS",Review and Refresh Content
3507,content_074ba6ead17b,5,"OLD_CONTENT, LOW_CTR, HIGH_IMPRESSIONS",Review and Refresh Content
3429,content_7b4e68b406b8,4,"OLD_CONTENT, LOW_CTR",Review and Refresh Content
1147,content_ab27c30d81f4,4,"OLD_CONTENT, LOW_CTR",Review and Refresh Content
23619,content_24abafed9707,4,"OLD_CONTENT, LOW_CTR",Review and Refresh Content
5273,content_9106162c6861,4,"OLD_CONTENT, LOW_CTR",Review and Refresh Content
19447,content_8bc10f396d2e,4,"OLD_CONTENT, LOW_CTR",Review and Refresh Content
15790,content_6476d1d8c050,4,"OLD_CONTENT, LOW_CTR",Review and Refresh Content
24305,content_d661e9eee4b6,4,"OLD_CONTENT, LOW_CTR",Review and Refresh Content


In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

## Top-20 Review

The top-ranked pages were selected because they combined high impressions, older content, and low CTR.

Action: Review and Refresh Content

Reason Codes:
- OLD_CONTENT
- LOW_CTR
- HIGH_IMPRESSIONS

Confidence:
Medium. These pages appear to be good refresh candidates based on observable signals.

What could make this recommendation wrong?
Some pages may already meet business goals, seasonal demand may have changed, or low CTR may not require a content refresh.

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

## Weak Picks

Some pages may receive a high score because they are old, even if they are still performing well.

## Leakage Check

No future information or product flags were used.

The rule only uses observable features:
- days_since_last_update
- impressions_90d
- ctr

No label-derived columns such as trend_direction or trend_pct were included.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.